In [1]:
import psycopg2

conn = psycopg2.connect(
        host="localhost",
        database="kenexaihackathon",
        user="postgres",
        password="09092002",
        port="5432"
    )

cur = conn.cursor()

In [9]:
# cSpell:ignore MERAKI meraki AUVIK auvik NCENTRAL ncentral

# Clear any previous failed transaction state in this notebook session
conn.rollback()

# ----------------------------
# MERAKI INGESTION
# ----------------------------
def ingest_meraki():
    cur.execute("""
    SELECT
        alert_id,
        correlation_id,
        organization_name,
        device_name,
        device_serial,
        check_name,
        alert_type,
        alert_level,
        event_state,
        occurred_at,
        synthetic,
        ingestion_time
    FROM bronze.meraki_alerts
    """)

    rows = cur.fetchall()

    for r in rows:
        cur.execute("""
        INSERT INTO silver.alerts
        (
            source_system,
            alert_id,
            correlation_id,
            organization_name,
            device_name,
            device_identifier,
            service_name,
            alert_type,
            severity,
            event_state,
            event_time,
            synthetic,
            ingestion_time
        )
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
        ON CONFLICT DO NOTHING
        """,
        (
            "meraki",
            r[0],
            r[1],
            r[2],
            r[3],
            r[4],
            r[5],
            r[6],
            r[7],
            r[8],
            r[9],
            r[10],
            r[11]
        ))


# ----------------------------
# AUVIK INGESTION
# ----------------------------
def ingest_auvik():
    cur.execute("""
    SELECT
        alert_id,
        correlation_id,
        company_name,
        entity_name,
        entity_id,
        entity_type,
        alert_name,
        alert_severity_string,
        event_state,
        event_time,
        synthetic,
        ingestion_time
    FROM bronze.auvik_alerts
    """)

    rows = cur.fetchall()

    for r in rows:
        cur.execute("""
        INSERT INTO silver.alerts
        (
            source_system,
            alert_id,
            correlation_id,
            organization_name,
            device_name,
            device_identifier,
            service_name,
            alert_type,
            severity,
            event_state,
            event_time,
            synthetic,
            ingestion_time
        )
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
        ON CONFLICT DO NOTHING
        """,
        (
            "auvik",
            r[0],
            r[1],
            r[2],
            r[3],
            r[4],
            r[5],
            r[6],
            r[7],
            r[8],
            r[9],
            r[10],
            r[11]
        ))


# ----------------------------
# NCENTRAL INGESTION
# ----------------------------
def ingest_ncentral():
    cur.execute("""
    SELECT
        active_notification_trigger_id,
        correlation_id,
        customer_name,
        device_name,
        device_uri,
        affected_service,
        qualitative_new_state,
        event_state,
        time_of_state_change,
        synthetic,
        ingestion_time
    FROM bronze.ncentral_alerts
    """)

    rows = cur.fetchall()

    for r in rows:
        cur.execute("""
        INSERT INTO silver.alerts
        (
            source_system,
            alert_id,
            correlation_id,
            organization_name,
            device_name,
            device_identifier,
            service_name,
            alert_type,
            severity,
            event_state,
            event_time,
            synthetic,
            ingestion_time
        )
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
        ON CONFLICT DO NOTHING
        """,
        (
            "ncentral",
            r[0],
            r[1],
            r[2],
            r[3],
            r[4],
            r[5],
            r[5],
            r[6],
            r[7],
            r[8],
            r[9],
            r[10]
        ))


# ----------------------------
# RUN PIPELINE
# ----------------------------
try:
    print("Ingesting Meraki alerts...")
    ingest_meraki()

    print("Ingesting Auvik alerts...")
    ingest_auvik()

    print("Ingesting NCentral alerts...")
    ingest_ncentral()

    conn.commit()
    print("Silver ingestion completed.")
except Exception as e:
    conn.rollback()
    print(f"Silver ingestion failed: {e}")
    raise


Ingesting Meraki alerts...
Ingesting Auvik alerts...
Ingesting NCentral alerts...
Silver ingestion completed.
